# CERTUS FLEX

```{device-card} gyger-certus-flex
```

**Hardware verified on C196069** (firmware 1.0.7.378-trunk, angled FLEX8, direct Ethernet): identity/status reads, plate motion, 0.2 bar regulation/discharge, timed water dispensing, native prime/purge, and the clean cycle. Water flow was observed from both connected channels. Volume-to-time conversion is checked against the vendor algorithm. See [verification scope](protocol.md#verification-scope) for the tested configuration and operations. Run this notebook one cell at a time.

| Property | Value |
| --- | --- |
| Connection | Direct controller UDP 1964; optional INCO server TCP 7979 |
| Heads | FLEX5, FLEX8, angled FLEX4, angled FLEX8 |
| Units | mm, seconds, µL; pressure explicitly in bar |
| Supported | Identity/status, reference/position/park, pressure, timed and calibrated valve dispensing, prime/purge/clean cycle |

## How it connects

PLR connects directly to the controller over Ethernet, without a vendor DLL or control PC. Put the wired interface on the controller's subnet. The tested controller used `192.168.1.3`, with the computer at `192.168.1.2/24`; confirm your machine's addresses. To use an existing INCO server instead, set `CERTUS_INCO_TARGET` to its configured target name and use the server PC's address as `CERTUS_HOST`. See [protocol and verification notes](protocol.md).

## Physical setup

Confirm the installed head type and channel labels. Set up the machine, compatible vessels, valves, tubing, compressed air supply, and waste collection according to its manual. Use water for the first fluid test. Ensure the axes and clamp have clearance. Finish any run in CERTUS CONTROL and use only one controlling application at a time.

Set `CERTUS_HOST` and `CERTUS_HEAD` in the notebook environment. Head values are `flex5`, `flex8`, `flex4_angled`, or `flex8_angled`. For direct Ethernet, leave `CERTUS_INCO_TARGET` unset. Optionally set `CERTUS_SOURCE_IP` to the local wired interface's IPv4 address.

## Configure

Constructing the object does not connect to the machine.

In [ ]:
import os
from pylabrobot.gyger import CertusFlex, ValveCalibration

certus = CertusFlex(
  host=os.environ["CERTUS_HOST"],
  target=os.environ.get("CERTUS_INCO_TARGET"),
  head=os.environ["CERTUS_HEAD"],
  source_ip=os.environ.get("CERTUS_SOURCE_IP"),
  timeout=60,
)

## Connect and capture identity

`setup()` only connects and reads serial number and firmware. Capture the first exchange for diagnosis; this cell does not request motion or pressure changes.

In [ ]:
from pylabrobot.io import start_capture, stop_capture

start_capture("certus-first-read")
try:
  await certus.setup()
  print(certus.serial_number, certus.firmware_version)
finally:
  stop_capture()

## Read status

Compare these states with the control software before attempting motion. Resolve machine faults using the normal operator procedure; the driver does not automatically acknowledge them.

In [ ]:
await certus.get_status()

## Read position

These are controller X/Y/Z coordinates in mm. They are not PLR deck or well coordinates.

In [ ]:
await certus.get_position()

## Read pressure

The result is actual dispensing pressure in bar.

In [ ]:
await certus.get_pressure()

## Initialize

Continue only after the read-only checks match the machine. Clear the workspace before running this cell: initialization can start the dispenser and reference its axes.

In [ ]:
await certus.initialize()

## Park

This raises Z to the configured park height before moving XY to park.

In [ ]:
await certus.park()

## Position over waste

Check that the configured prime location is over the waste collector. This moves Z to park, XY to prime, and Z to prime. It does not open a valve.

In [ ]:
await certus.move_to_prime()

## Set pressure

Confirm that the connected vessels and installed valves are suitable for 0.2 bar, or edit the value to the verified operating pressure. This setting was exercised with water; it is not a universal valve default. The cell pressurizes the supply and waits for the measured pressure.

In [ ]:
await certus.set_pressure(0.2)

## Deliver a timed test pulse

Confirm channel 1 is prepared with water and aimed into waste. `valves[0]` is the first printed head channel. This requests one 1 ms pulse and waits for completion. A pulse is not a known volume without calibration. Do not rerun a failed dispense automatically; it may already have delivered liquid.

In [ ]:
await certus.valves[0].dispense_for(0.001)

## Optional: measured volume calibration

Skip this section until a calibration has been measured for this exact valve, fluid, and pressure. Prepare `measured-calibration.json` with `pressure_bar` and a `points` list of `[volume_in_uL, duration_in_seconds]` pairs. Both values must increase, with at least two points. This cell only loads calibration; it does not dispense.

In [ ]:
import json
from pathlib import Path

measured = json.loads(Path("measured-calibration.json").read_text())
certus.valves[0].calibration = ValveCalibration(
  points=tuple(tuple(point) for point in measured["points"]),
  pressure_bar=measured["pressure_bar"],
)

## Optional: dispense a calibrated volume

Keep the valve aimed into waste for validation. Enter a volume within the measured calibration range and confirm the measured pressure matches the calibration. This cell delivers liquid.

The optional `correction_factor` argument matches the channel volume correction in CERTUS CONTROL (0.5–1.5, default 1.0). It multiplies the requested volume before interpolation. Use the factor for your setup; PLR does not read it from the vendor application. Physical volume accuracy comes from the valve/fluid calibration and is separate from verifying the driver's calculation.

In [ ]:
volume = float(input("Verified test volume in µL: "))
await certus.valves[0].dispense(volume)

## Prime a channel

The head must be over waste and pressure regulated. This opens channel 1 continuously for 100 ms using the native Prime command and waits for completion. The controller's prime timeout is checked before opening. A timed opening is not a calibrated priming volume.

In [ ]:
await certus.valves[0].prime(0.1)

## Purge a channel

Keep the head over waste. This requests 20 ms of cumulative opening time as 1 ms pulses. Pulse timing is enforced by firmware; total elapsed time also includes the firmware's inter-pulse waits.

In [ ]:
await certus.valves[0].purge(0.02, pulse_duration=0.001)

## Clean cycle

Confirm waste collection and select the prepared channels. Indexes `[1, 2]` mean physical channels 2 and 3. This moves to the purge position, purges each channel, waits 250 ms, primes each channel, and parks. It performs one cycle at the existing pressure. These timings test functionality; they are not a validated cleaning recipe. Discharge is explicit in `finally`. This one-cycle sequence is hardware-verified with water on channels 2 and 3 at 0.2 bar.

In [ ]:
try:
  await certus.clean_cycle(
    channels=[1, 2],
    prime_duration=0.1,
    purge_duration=0.02,
    purge_pulse_duration=0.001,
    cycles=1,
  )
finally:
  await certus.set_pressure(0)

## Discharge pressure

Explicitly discharge the vessels before disconnecting. Disconnecting alone does not change machine state.

In [ ]:
await certus.set_pressure(0)

## Disconnect

This closes the connection. It does not abort motion, stop a queued pulse, or operate an emergency stop.

In [ ]:
await certus.stop()